In [1]:
import json
import pandas as pd
from tqdm import tqdm
from requests import get
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
limit = 100
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36",
    "Content-Type": "application/json",
}

url_template = (
    "https://maganghub.kemnaker.go.id/be/v1/api/list/vacancies-aktif?"
    "order_by=jumlah_terdaftar&order_direction=ASC&page={page}&limit={limit}"
)

all_data = []


def fetch_page(page):
    while True:
        try:
            res = get(
                url_template.format(page=page, limit=limit), headers=headers
            )
            if res.status_code == 200:
                data = json.loads(res.text)
                return data
        except:
            pass


def fetch_all():
    first_page_data = fetch_page(1)
    if not first_page_data or "meta" not in first_page_data:
        print("Gagal mendapatkan metadata halaman pertama.")
        return []

    meta = first_page_data["meta"]
    last_page = meta.get("pagination", {}).get("last_page", 1)

    print(f"📄 Total halaman: {last_page}")

    results = []
    results.extend(first_page_data.get("data", []))

    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {
            executor.submit(fetch_page, page): page for page in range(2, last_page + 1)
        }

        for future in tqdm(
            as_completed(futures), total=len(futures), desc="Mengunduh data"
        ):
            page = futures[future]
            try:
                page_data = future.result()
                if page_data and "data" in page_data:
                    results.extend(page_data["data"])
            except Exception as e:
                print(f"❌ Gagal ambil halaman {page}: {e}")

    return results


all_data = fetch_all()
print(f"\n✅ Total data terkumpul: {len(all_data)}")

📄 Total halaman: 379


Mengunduh data: 100%|██████████| 378/378 [06:25<00:00,  1.02s/it]



✅ Total data terkumpul: 37856


In [3]:
df = pd.DataFrame(all_data)

df.to_json(
    "raw_data.json", indent=4, orient="records", date_format="iso", date_unit="s"
)

df.head(5)

,id_posisi,posisi,deskripsi_posisi,syarat_khusus,usia_minimal,usia_maksimal,revisi,id_kondisi_fisik,id_program,created_at,...,jumlah_kuota,jumlah_terdaftar,program_studi,jenjang,ref_status_posisi,program,perusahaan,jadwal,government_agency,sub_government_agency
0,a0445e58-710f-4b07-872b-d77b9a24b0b1,DOKTER UMUM,Tugas dokter umum di rumah sakit meliputi peme...,None,None,None,None,None,f6268d0a-cf4a-4ff3-ad11-6a1bf4ada866,2025-11-03 13:53:25,...,5,0,"[{""id"":""7428ac9a-c11e-4909-ac0f-56865c24b0e5"",...","[""Sarjana""]","{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': 'f6268d0a-cf4a-4ff3-ad11-6a1bf4...,{'id_perusahaan': 'ae20e495-458a-4fd0-b23f-f1c...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,{'government_agency_name': None},{'sub_government_agency_name': None}
1,a0540e70-79d9-4feb-be84-49d2a786a375,Perawat Kesehatan,1. Memberikan perawatan kesehatan umum dan tin...,None,None,None,None,None,ef448d10-a0e0-4c47-9047-4f7893cd3c55,2025-11-11 08:46:26,...,2,0,"[{""id"":""dbcc16ac-6297-4f5e-a1f8-d6804bb972c7"",...","[""Sarjana""]","{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': 'ef448d10-a0e0-4c47-9047-4f7893...,{'id_perusahaan': '0cca1976-5bd2-4b62-b226-7f0...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,{'government_agency_name': 'Kementerian Imigra...,{'sub_government_agency_name': 'LEMBAGA PEMASY...
2,a052e840-8d00-4d5b-8a0a-8c760a2b12a7,Pembina Kepribadian,1. Menyusun dan melaksanakan program\npembinaa...,None,None,None,None,None,71704e63-55ce-4431-ba6d-e52637d73fce,2025-11-10 07:30:19,...,2,0,"[{""id"":""18bd7010-64b2-41b6-94bb-0e6c092b742f"",...","[""Sarjana""]","{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': '71704e63-55ce-4431-ba6d-e52637...,{'id_perusahaan': '1d1690e9-5f1b-4159-bab9-add...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,{'government_agency_name': 'Kementerian Imigra...,{'sub_government_agency_name': 'RUMAH TAHANAN ...
3,a0445f94-fb96-4968-8a3d-8ad03658dc04,DOKTER GIGI,Tugas dokter gigi di rumah sakit meliputi peme...,None,None,None,None,None,f6268d0a-cf4a-4ff3-ad11-6a1bf4ada866,2025-11-03 13:53:25,...,3,0,"[{""id"":""3bd4a71c-d764-4a0e-82ee-228e6d232a4c"",...","[""Sarjana""]","{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': 'f6268d0a-cf4a-4ff3-ad11-6a1bf4...,{'id_perusahaan': 'ae20e495-458a-4fd0-b23f-f1c...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,{'government_agency_name': None},{'sub_government_agency_name': None}
4,a0562831-2496-41d1-a1a9-1be8534533e3,Psikiater,1. Menangani gangguan kesehatan jiwa warga bin...,None,None,None,None,None,38b7e2d3-d51d-49dd-8972-caa926ffd3a6,2025-11-12 09:54:15,...,4,0,"[{""id"":""7428ac9a-c11e-4909-ac0f-56865c24b0e5"",...","[""Sarjana""]","{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': '38b7e2d3-d51d-49dd-8972-caa926...,{'id_perusahaan': 'bb6c71c5-4be5-4aa7-bee8-1a9...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,{'government_agency_name': 'Kementerian Imigra...,{'sub_government_agency_name': 'LEMBAGA PEMASY...


In [4]:
df = df.drop_duplicates(subset=["id_posisi"]).reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33503 entries, 0 to 33502
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_posisi              33503 non-null  object
 1   posisi                 33503 non-null  object
 2   deskripsi_posisi       33503 non-null  object
 3   syarat_khusus          0 non-null      object
 4   usia_minimal           0 non-null      object
 5   usia_maksimal          0 non-null      object
 6   revisi                 0 non-null      object
 7   id_kondisi_fisik       0 non-null      object
 8   id_program             33503 non-null  object
 9   created_at             33493 non-null  object
 10  updated_at             31298 non-null  object
 11  kode_kbji              0 non-null      object
 12  penempatan             0 non-null      object
 13  id_jenis_kelamin       0 non-null      object
 14  id_status_posisi       33503 non-null  int64 
 15  jumlah_kuota       

In [5]:
df = df.dropna(axis=1, how="all")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33503 entries, 0 to 33502
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_posisi              33503 non-null  object
 1   posisi                 33503 non-null  object
 2   deskripsi_posisi       33503 non-null  object
 3   id_program             33503 non-null  object
 4   created_at             33493 non-null  object
 5   updated_at             31298 non-null  object
 6   id_status_posisi       33503 non-null  int64 
 7   jumlah_kuota           33503 non-null  int64 
 8   jumlah_terdaftar       33503 non-null  int64 
 9   program_studi          33503 non-null  object
 10  jenjang                33503 non-null  object
 11  ref_status_posisi      33503 non-null  object
 12  program                33503 non-null  object
 13  perusahaan             33503 non-null  object
 14  jadwal                 33503 non-null  object
 15  government_agency  

In [6]:
df.columns

Index(['id_posisi', 'posisi', 'deskripsi_posisi', 'id_program', 'created_at',
       'updated_at', 'id_status_posisi', 'jumlah_kuota', 'jumlah_terdaftar',
       'program_studi', 'jenjang', 'ref_status_posisi', 'program',
       'perusahaan', 'jadwal', 'government_agency', 'sub_government_agency'],
      dtype='object')

In [7]:
df[["created_at", "updated_at"]] = pd.to_datetime(
    df[["created_at", "updated_at"]].stack()
).unstack()

df["kabupaten"] = df["perusahaan"].apply(lambda x: x["nama_kabupaten"])
df["provinsi"] = df["perusahaan"].apply(lambda x: x["nama_provinsi"])

df["diff_quota"] = df["jumlah_kuota"] - df["jumlah_terdaftar"]
df = df[[df.columns[-1]] + df.columns[:-1].tolist()]

df.head(5)

,diff_quota,id_posisi,posisi,deskripsi_posisi,id_program,created_at,updated_at,id_status_posisi,jumlah_kuota,jumlah_terdaftar,program_studi,jenjang,ref_status_posisi,program,perusahaan,jadwal,government_agency,sub_government_agency,kabupaten,provinsi
0,5,a0445e58-710f-4b07-872b-d77b9a24b0b1,DOKTER UMUM,Tugas dokter umum di rumah sakit meliputi peme...,f6268d0a-cf4a-4ff3-ad11-6a1bf4ada866,2025-11-03 13:53:25,2025-11-03 13:57:43,2,5,0,"[{""id"":""7428ac9a-c11e-4909-ac0f-56865c24b0e5"",...","[""Sarjana""]","{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': 'f6268d0a-cf4a-4ff3-ad11-6a1bf4...,{'id_perusahaan': 'ae20e495-458a-4fd0-b23f-f1c...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,{'government_agency_name': None},{'sub_government_agency_name': None},KAB. JEMBER,JAWA TIMUR
1,2,a0540e70-79d9-4feb-be84-49d2a786a375,Perawat Kesehatan,1. Memberikan perawatan kesehatan umum dan tin...,ef448d10-a0e0-4c47-9047-4f7893cd3c55,2025-11-11 08:46:26,2025-11-11 08:46:26,2,2,0,"[{""id"":""dbcc16ac-6297-4f5e-a1f8-d6804bb972c7"",...","[""Sarjana""]","{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': 'ef448d10-a0e0-4c47-9047-4f7893...,{'id_perusahaan': '0cca1976-5bd2-4b62-b226-7f0...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,{'government_agency_name': 'Kementerian Imigra...,{'sub_government_agency_name': 'LEMBAGA PEMASY...,KAB. TEGAL,JAWA TENGAH
2,2,a052e840-8d00-4d5b-8a0a-8c760a2b12a7,Pembina Kepribadian,1. Menyusun dan melaksanakan program\npembinaa...,71704e63-55ce-4431-ba6d-e52637d73fce,2025-11-10 07:30:19,2025-11-10 07:30:19,2,2,0,"[{""id"":""18bd7010-64b2-41b6-94bb-0e6c092b742f"",...","[""Sarjana""]","{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': '71704e63-55ce-4431-ba6d-e52637...,{'id_perusahaan': '1d1690e9-5f1b-4159-bab9-add...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,{'government_agency_name': 'Kementerian Imigra...,{'sub_government_agency_name': 'RUMAH TAHANAN ...,KAB. JEMBRANA,BALI
3,3,a0445f94-fb96-4968-8a3d-8ad03658dc04,DOKTER GIGI,Tugas dokter gigi di rumah sakit meliputi peme...,f6268d0a-cf4a-4ff3-ad11-6a1bf4ada866,2025-11-03 13:53:25,2025-11-03 13:57:43,2,3,0,"[{""id"":""3bd4a71c-d764-4a0e-82ee-228e6d232a4c"",...","[""Sarjana""]","{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': 'f6268d0a-cf4a-4ff3-ad11-6a1bf4...,{'id_perusahaan': 'ae20e495-458a-4fd0-b23f-f1c...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,{'government_agency_name': None},{'sub_government_agency_name': None},KAB. JEMBER,JAWA TIMUR
4,4,a0562831-2496-41d1-a1a9-1be8534533e3,Psikiater,1. Menangani gangguan kesehatan jiwa warga bin...,38b7e2d3-d51d-49dd-8972-caa926ffd3a6,2025-11-12 09:54:15,2025-11-12 09:54:15,2,4,0,"[{""id"":""7428ac9a-c11e-4909-ac0f-56865c24b0e5"",...","[""Sarjana""]","{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': '38b7e2d3-d51d-49dd-8972-caa926...,{'id_perusahaan': 'bb6c71c5-4be5-4aa7-bee8-1a9...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,{'government_agency_name': 'Kementerian Imigra...,{'sub_government_agency_name': 'LEMBAGA PEMASY...,KOTA PEKANBARU,RIAU


In [8]:
df["government_agency"] = df["government_agency"].apply(
    lambda x: x["government_agency_name"] if isinstance(x, dict) else None
)
df["sub_government_agency"] = df["sub_government_agency"].apply(
    lambda x: x["sub_government_agency_name"] if isinstance(x, dict) else None
)
df["program_studi"] = df["program_studi"].apply(
    lambda x: (
        ", ".join([i["title"] for i in json.loads(x)]) if isinstance(x, str) else None
    )
)
df["jenjang"] = df["jenjang"].apply(
    lambda x: ", ".join([i for i in eval(x)]) if isinstance(x, str) else None
)

df.head(5)

,diff_quota,id_posisi,posisi,deskripsi_posisi,id_program,created_at,updated_at,id_status_posisi,jumlah_kuota,jumlah_terdaftar,program_studi,jenjang,ref_status_posisi,program,perusahaan,jadwal,government_agency,sub_government_agency,kabupaten,provinsi
0,5,a0445e58-710f-4b07-872b-d77b9a24b0b1,DOKTER UMUM,Tugas dokter umum di rumah sakit meliputi peme...,f6268d0a-cf4a-4ff3-ad11-6a1bf4ada866,2025-11-03 13:53:25,2025-11-03 13:57:43,2,5,0,Kedokteran,Sarjana,"{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': 'f6268d0a-cf4a-4ff3-ad11-6a1bf4...,{'id_perusahaan': 'ae20e495-458a-4fd0-b23f-f1c...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,None,None,KAB. JEMBER,JAWA TIMUR
1,2,a0540e70-79d9-4feb-be84-49d2a786a375,Perawat Kesehatan,1. Memberikan perawatan kesehatan umum dan tin...,ef448d10-a0e0-4c47-9047-4f7893cd3c55,2025-11-11 08:46:26,2025-11-11 08:46:26,2,2,0,Ilmu Gizi,Sarjana,"{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': 'ef448d10-a0e0-4c47-9047-4f7893...,{'id_perusahaan': '0cca1976-5bd2-4b62-b226-7f0...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,Kementerian Imigrasi dan Pemasyarakatan,LEMBAGA PEMASYARAKATAN KELAS IIB SLAWI,KAB. TEGAL,JAWA TENGAH
2,2,a052e840-8d00-4d5b-8a0a-8c760a2b12a7,Pembina Kepribadian,1. Menyusun dan melaksanakan program\npembinaa...,71704e63-55ce-4431-ba6d-e52637d73fce,2025-11-10 07:30:19,2025-11-10 07:30:19,2,2,0,Pendidikan Agama Islam,Sarjana,"{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': '71704e63-55ce-4431-ba6d-e52637...,{'id_perusahaan': '1d1690e9-5f1b-4159-bab9-add...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,Kementerian Imigrasi dan Pemasyarakatan,RUMAH TAHANAN NEGARA KELAS IIB NEGARA,KAB. JEMBRANA,BALI
3,3,a0445f94-fb96-4968-8a3d-8ad03658dc04,DOKTER GIGI,Tugas dokter gigi di rumah sakit meliputi peme...,f6268d0a-cf4a-4ff3-ad11-6a1bf4ada866,2025-11-03 13:53:25,2025-11-03 13:57:43,2,3,0,Kedokteran Gigi,Sarjana,"{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': 'f6268d0a-cf4a-4ff3-ad11-6a1bf4...,{'id_perusahaan': 'ae20e495-458a-4fd0-b23f-f1c...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,None,None,KAB. JEMBER,JAWA TIMUR
4,4,a0562831-2496-41d1-a1a9-1be8534533e3,Psikiater,1. Menangani gangguan kesehatan jiwa warga bin...,38b7e2d3-d51d-49dd-8972-caa926ffd3a6,2025-11-12 09:54:15,2025-11-12 09:54:15,2,4,0,Kedokteran,Sarjana,"{'id_status_posisi': 2, 'nama_status_posisi': ...",{'id_program': '38b7e2d3-d51d-49dd-8972-caa926...,{'id_perusahaan': 'bb6c71c5-4be5-4aa7-bee8-1a9...,{'id_jadwal': '48039861-1251-474e-86af-d904e82...,Kementerian Imigrasi dan Pemasyarakatan,LEMBAGA PEMASYARAKATAN NARKOTIKA KELAS IIB RUMBAI,KOTA PEKANBARU,RIAU


In [9]:
df.to_json("data.json", indent=4, orient="records", date_format="iso", date_unit="s")